In [1]:
import pandas as pd
import numpy as np
import random

# Settings
NUM_SAMPLES = 1500  # Size of dataset (good for college projects)

# 1. Define Categories based on OSMI & Your Project Requirements
genders = ['Male', 'Female', 'Non-Binary']
yes_no = ['Yes', 'No']
work_pressure_levels = ['Low', 'Medium', 'High']
sleep_quality = ['Good', 'Average', 'Poor']

# 2. Generate Base Data
data = {
    'Age': np.random.normal(30, 7, NUM_SAMPLES).astype(int),  # Mean age 30
    'Gender': np.random.choice(genders, NUM_SAMPLES, p=[0.55, 0.40, 0.05]),
    'Family_History': np.random.choice(yes_no, NUM_SAMPLES, p=[0.4, 0.6]),
    'Work_Pressure': np.random.choice(work_pressure_levels, NUM_SAMPLES),
    'Financial_Stress': np.random.choice(yes_no, NUM_SAMPLES, p=[0.45, 0.55]),
    'Anonymity_Protected': np.random.choice(yes_no, NUM_SAMPLES), # New useful feature
    'Sleep_Issues': []  # Will be calculated based on pressure
}

# 3. Add Logical Correlations (To ensure the ML model can actually learn)
treatment_needed = []

for i in range(NUM_SAMPLES):
    score = 0
    
    # Feature 1: Work Pressure affects Sleep
    pressure = data['Work_Pressure'][i]
    if pressure == 'High':
        sleep = np.random.choice(['Poor', 'Average'], p=[0.7, 0.3])
        score += 3
    elif pressure == 'Medium':
        sleep = np.random.choice(['Average', 'Poor', 'Good'], p=[0.5, 0.2, 0.3])
        score += 1
    else:
        sleep = 'Good'
    
    data['Sleep_Issues'].append(sleep)
    
    # Feature 2: Family History adds risk
    if data['Family_History'][i] == 'Yes':
        score += 3
        
    # Feature 3: Financial Stress adds risk
    if data['Financial_Stress'][i] == 'Yes':
        score += 2
        
    # Feature 4: Sleep Issues significantly add risk
    if sleep == 'Poor':
        score += 3
    
    # Feature 5: Gender factor (Statistical bias in surveys: Females often report more)
    if data['Gender'][i] == 'Female':
        score += 1
        
    # Final Decision: Threshold for "Treatment Needed"
    # Add some randomness so it's not a perfect rule (simulating real life)
    total_score = score + np.random.randint(-1, 2)
    
    if total_score >= 5:
        treatment_needed.append('Yes')
    else:
        treatment_needed.append('No')

data['Treatment_Needed'] = treatment_needed

# 4. Save to CSV
df = pd.DataFrame(data)

# Clip Age to realistic bounds (e.g., 18 to 60)
df['Age'] = df['Age'].clip(18, 60)

df.to_csv('mental_health_final_dataset.csv', index=False)
print("✅ Optimal Dataset 'mental_health_final_dataset.csv' created with 1500 rows.")
print(df.head())

✅ Optimal Dataset 'mental_health_final_dataset.csv' created with 1500 rows.
   Age      Gender Family_History Work_Pressure Financial_Stress  \
0   27      Female             No        Medium               No   
1   30        Male             No          High               No   
2   24      Female             No          High               No   
3   28      Female            Yes           Low              Yes   
4   35  Non-Binary             No        Medium               No   

  Anonymity_Protected Sleep_Issues Treatment_Needed  
0                  No         Good               No  
1                  No         Poor              Yes  
2                  No         Poor              Yes  
3                  No         Good              Yes  
4                 Yes         Good               No  


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
import joblib

In [2]:
# ---------------------------------------------------------
# 1. LOAD DATA
# ---------------------------------------------------------
df = pd.read_csv('survey.csv')

In [3]:
# ---------------------------------------------------------
# 2. DATA CLEANING (CRITICAL STEP)
# ---------------------------------------------------------

# A. Clean 'Age': Filter out outliers (negative ages, or > 100)
df.drop(df[df['Age'] < 18].index, inplace=True)
df.drop(df[df['Age'] > 100].index, inplace=True)

# B. Clean 'Gender': Standardize the messy inputs into 3 categories
# This mapping handles common variations found in OSMI 2014
def clean_gender(gender):
    gender = str(gender).lower().strip()
    if gender in ['male', 'm', 'man', 'cis male', 'male-ish', 'maile', 'mal', 'male (cis)', 'make', 'msle']:
        return 'Male'
    elif gender in ['female', 'f', 'woman', 'cis female', 'femake', 'female (cis)', 'women']:
        return 'Female'
    else:
        return 'Other' # Covers non-binary, queer, trans, fluid, etc.

df['Gender'] = df['Gender'].apply(clean_gender)

# C. Handle Missing Values
# 'work_interfere' has many NaNs. We fill them with "Unknown" so we don't lose data.
# This column is a STRONG predictor, so we must keep it.
df['work_interfere'] = df['work_interfere'].fillna('Unknown')
df['self_employed'] = df['self_employed'].fillna('No')

# D. Drop columns we don't need for prediction (Text comments, Timestamp)
cols_to_drop = ['Timestamp', 'comments', 'state', 'Country']
df = df.drop(columns=cols_to_drop)

In [4]:
# ---------------------------------------------------------
# 3. ENCODING (Convert Text to Numbers)
# ---------------------------------------------------------
# We use LabelEncoders and save them so the Backend can use them later
encoders = {}
target_col = 'treatment' 

# Identify categorical columns (everything except Age)
cat_cols = [c for c in df.columns if c not in ['Age']]

for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    encoders[col] = le

In [5]:
# ---------------------------------------------------------
# 4. TRAINING
# ---------------------------------------------------------
X = df.drop(target_col, axis=1)
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Random Forest: The standard for this dataset
model = RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42)
model.fit(X_train, y_train)

RandomForestClassifier(max_depth=15, random_state=42)

In [6]:
# ---------------------------------------------------------
# 5. EVALUATION
# ---------------------------------------------------------
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"✅ Model Trained successfully!")
print(f"📊 Accuracy on Test Data: {accuracy * 100:.2f}%")

# Show Feature Importance (What matters most?)
importances = pd.Series(model.feature_importances_, index=X.columns)
print("\nTop 5 Predictors:")
print(importances.nlargest(5))

✅ Model Trained successfully!
📊 Accuracy on Test Data: 80.08%

Top 5 Predictors:
work_interfere    0.304308
Age               0.086503
family_history    0.078326
care_options      0.056724
no_employees      0.047674
dtype: float64


In [7]:
# ---------------------------------------------------------
# 6. SAVE ARTIFACTS
# ---------------------------------------------------------
joblib.dump(model, 'mental_health_osmi_model.pkl')
joblib.dump(encoders, 'osmi_encoders.pkl')
print("\n💾 Model & Encoders saved to disk.")


💾 Model & Encoders saved to disk.


## Inference

In [8]:
import pandas as pd
import joblib
import numpy as np

# ---------------------------------------------------------
# 1. LOAD ARTIFACTS
# ---------------------------------------------------------
# Load the trained model
model = joblib.load('mental_health_osmi_model.pkl')
# Load the encoders used during training
encoders = joblib.load('osmi_encoders.pkl')

print("✅ Model and Encoders loaded successfully.")

# ---------------------------------------------------------
# 2. DEFINE INPUT DATA
# ---------------------------------------------------------
# This simulates a user filling out the form on your website.
# Note: The keys MUST match the column names from your training data exactly.
new_user_input = {
    'Age': 25,
    'Gender': 'Male',
    'self_employed': 'No',
    'family_history': 'Yes',
    'work_interfere': 'Rarely',  # Options: 'Often', 'Rarely', 'Never', 'Sometimes', 'Unknown'
    'no_employees': '6-25',
    'remote_work': 'No',
    'tech_company': 'Yes',
    'benefits': 'Yes',
    'care_options': 'Not sure',
    'wellness_program': 'No',
    'seek_help': 'No',
    'anonymity': 'Yes',
    'leave': 'Somewhat easy',
    'mental_health_consequence': 'No',
    'phys_health_consequence': 'No',
    'coworkers': 'Some of them',
    'supervisor': 'Yes',
    'mental_health_interview': 'No',
    'phys_health_interview': 'Maybe',
    'mental_vs_physical': 'Yes',
    'obs_consequence': 'No'
}

# Convert dictionary to DataFrame (1 row)
input_df = pd.DataFrame([new_user_input])

# ---------------------------------------------------------
# 3. PREPROCESSING (Re-apply the same logic!)
# ---------------------------------------------------------
# We must encode the text inputs (e.g., 'Male' -> 0) using the saved encoders.

# Identify which columns need encoding (same list as training)
# IMPORTANT: We only encode columns that we actually have encoders for.
for col, encoder in encoders.items():
    if col in input_df.columns:
        # Handle unseen labels (e.g., if user types something new)
        # For simplicity in this test, we assume inputs are valid standard options
        try:
            input_df[col] = encoder.transform(input_df[col])
        except ValueError:
            # Fallback for unknown categories (assign to most common or 0)
            print(f"⚠️ Warning: Unseen label in {col}, setting to 0")
            input_df[col] = 0

# ---------------------------------------------------------
# 4. PREDICTION
# ---------------------------------------------------------
# Make the prediction
prediction = model.predict(input_df)
probability = model.predict_proba(input_df)

# Decode the target (0 or 1) back to "Yes" or "No" if needed
# In OSMI dataset: usually 'Yes' mapped to 1, 'No' mapped to 0
result_text = "Treatment Needed" if prediction[0] == 1 else "No Treatment Needed"
confidence_score = probability[0][prediction[0]] * 100

print("\n------------------------------------------------")
print(f"👤 User Input Summary: Age {new_user_input['Age']}, Gender {new_user_input['Gender']}")
print(f"🧠 Prediction: {result_text}")
print(f"📊 Confidence: {confidence_score:.2f}%")
print("------------------------------------------------")

✅ Model and Encoders loaded successfully.

------------------------------------------------
👤 User Input Summary: Age 25, Gender Male
🧠 Prediction: Treatment Needed
📊 Confidence: 74.40%
------------------------------------------------


In [ ]:
import pandas as pd
import joblib
import numpy as np
import sys

# ---------------------------------------------------------
# HELPER FUNCTION FOR INTERACTIVE INPUT
# ---------------------------------------------------------
def get_user_input(question, options=None, input_type=str):
    """
    Asks the user a question and validates the input.
    - question: String to display
    - options: List of valid strings (e.g., ['Yes', 'No'])
    - input_type: Type of input expected (int or str)
    """
    while True:
        try:
            if options:
                print(f"\n❓ {question}")
                print(f"   Options: [{', '.join(options)}]")
                user_response = input("   👉 Your Answer: ").strip()
                
                # Case-insensitive matching for better UX
                match = next((opt for opt in options if opt.lower() == user_response.lower()), None)
                if match:
                    return match
                else:
                    print("   ❌ Invalid option. Please type exactly one of the options above.")
            else:
                # Free text or number input
                user_response = input(f"\n❓ {question}\n   👉 Your Answer: ").strip()
                if input_type == int:
                    if not user_response.isdigit():
                        raise ValueError("Please enter a valid number.")
                    return int(user_response)
                return user_response

        except ValueError as e:
            print(f"   ❌ Error: {e}")

# ---------------------------------------------------------
# 1. LOAD ARTIFACTS
# ---------------------------------------------------------
try:
    print("⏳ Loading model artifacts...")
    model = joblib.load('mental_health_osmi_model.pkl')
    encoders = joblib.load('osmi_encoders.pkl')
    print("✅ Model and Encoders loaded successfully.")
except FileNotFoundError:
    print("❌ Error: Model files not found. Run 'train_model_osmi.py' first.")
    sys.exit()

# ---------------------------------------------------------
# 2. COLLECT USER INPUT INTERACTIVELY
# ---------------------------------------------------------
print("\n=======================================================")
print("🧠 MENTAL HEALTH PREDICTION - INTERACTIVE MODE")
print("   Please answer the following questions honestly.")
print("=======================================================")

user_data = {}

# --- Personal Information ---
user_data['Age'] = get_user_input("What is your age?", input_type=int)
user_data['Gender'] = get_user_input("What is your gender?", options=['Male', 'Female', 'Other'])
user_data['family_history'] = get_user_input("Do you have a family history of mental illness?", options=['Yes', 'No'])
user_data['work_interfere'] = get_user_input("Does your mental health interfere with your work?", options=['Often', 'Rarely', 'Never', 'Sometimes', 'Unknown'])

# --- Work Environment ---
user_data['self_employed'] = get_user_input("Are you self-employed?", options=['Yes', 'No'])
user_data['no_employees'] = get_user_input("How many employees does your company have?", options=['1-5', '6-25', '26-100', '100-500', '500-1000', 'More than 1000'])
user_data['remote_work'] = get_user_input("Do you work remotely at least 50% of the time?", options=['Yes', 'No'])
user_data['tech_company'] = get_user_input("Is your employer primarily a tech company?", options=['Yes', 'No'])
user_data['benefits'] = get_user_input("Does your employer provide mental health benefits?", options=['Yes', 'No', "Don't know"])
user_data['care_options'] = get_user_input("Do you know the options for mental health care your employer provides?", options=['Yes', 'No', 'Not sure'])
user_data['wellness_program'] = get_user_input("Has your employer discussed mental health as part of a wellness program?", options=['Yes', 'No', "Don't know"])
user_data['seek_help'] = get_user_input("Does your employer provide resources to learn more about mental health issues?", options=['Yes', 'No', "Don't know"])
user_data['anonymity'] = get_user_input("Is your anonymity protected if you use mental health resources?", options=['Yes', 'No', "Don't know"])
user_data['leave'] = get_user_input("How easy is it for you to take medical leave for a mental health condition?", options=['Very easy', 'Somewhat easy', 'Somewhat difficult', 'Very difficult', "Don't know"])

# --- Social & Stigma ---
user_data['mental_health_consequence'] = get_user_input("Do you think that discussing a mental health issue with your employer would have negative consequences?", options=['Yes', 'No', 'Maybe'])
user_data['phys_health_consequence'] = get_user_input("Do you think that discussing a physical health issue with your employer would have negative consequences?", options=['Yes', 'No', 'Maybe'])
user_data['coworkers'] = get_user_input("Would you be willing to discuss a mental health issue with your coworkers?", options=['Yes', 'No', 'Some of them'])
user_data['supervisor'] = get_user_input("Would you be willing to discuss a mental health issue with your direct supervisor?", options=['Yes', 'No', 'Some of them'])
user_data['mental_health_interview'] = get_user_input("Would you bring up a mental health issue with a potential employer in an interview?", options=['Yes', 'No', 'Maybe'])
user_data['phys_health_interview'] = get_user_input("Would you bring up a physical health issue with a potential employer in an interview?", options=['Yes', 'No', 'Maybe'])
user_data['mental_vs_physical'] = get_user_input("Do you feel that your employer takes mental health as seriously as physical health?", options=['Yes', 'No', "Don't know"])
user_data['obs_consequence'] = get_user_input("Have you heard of or observed negative consequences for coworkers for their mental health conditions?", options=['Yes', 'No'])


# Convert dictionary to DataFrame (1 row)
input_df = pd.DataFrame([user_data])

# ---------------------------------------------------------
# 3. PREPROCESSING
# ---------------------------------------------------------
# Identify which columns need encoding (same list as training)
for col, encoder in encoders.items():
    if col in input_df.columns:
        try:
            # Normalize Gender specifically before encoding
            if col == 'Gender':
                val = input_df[col].iloc[0].lower().strip()
                if val in ['male', 'm', 'man', 'cis male']: normalized_val = 'Male'
                elif val in ['female', 'f', 'woman']: normalized_val = 'Female'
                else: normalized_val = 'Other'
                input_df[col] = normalized_val

            # Transform input
            input_df[col] = encoder.transform([input_df[col].iloc[0]])
        except ValueError:
            # Fallback for unknown categories (assign to 0 - usually the first class)
            # This prevents crashing if an odd value slips through
            print(f"⚠️ Warning: Unseen label in {col}, defaulting to 0")
            input_df[col] = 0

# ---------------------------------------------------------
# 4. PREDICTION
# ---------------------------------------------------------
print("\n🔄 Processing answers...")

# The model expects columns in a specific order. We reorder input_df to match.
if hasattr(model, "feature_names_in_"):
    input_df = input_df[model.feature_names_in_]
    
    
prediction = model.predict(input_df)
probability = model.predict_proba(input_df)

result_text = "Treatment Needed" if prediction[0] == 1 else "No Treatment Needed"
confidence_score = probability[0][prediction[0]] * 100

# ---------------------------------------------------------
# 5. FINAL OUTPUT
# ---------------------------------------------------------
print("\n" + "="*50)
print(f"📊 REPORT GENERATED FOR USER (Age: {user_data['Age']})")
print("="*50)

if prediction[0] == 1:
    print(f"🔴 RESULT: {result_text}")
    print("   Based on your responses, we recommend consulting a mental health professional.")
else:
    print(f"🟢 RESULT: {result_text}")
    print("   Your responses suggest you are currently managing well.")

print(f"\n   (Model Confidence: {confidence_score:.2f}%)")
print("="*50 + "\n")

⏳ Loading model artifacts...
✅ Model and Encoders loaded successfully.

🧠 MENTAL HEALTH PREDICTION - INTERACTIVE MODE
   Please answer the following questions honestly.
